In [37]:
import oracledb
try:
    conn = oracledb.connect(
        user="hr",
        password="basededatos",
        dsn="localhost:1521/XE"  
    )
    cursor = conn.cursor()
    cursor.callproc("dbms_output.enable")


except oracledb.Error as e:
    print(f"Error detectado: {e}")

# UNIVERSIDAD DE LAS FUERZAS ARMADAS "ESPE"
## Elaborado por: Camila Bohórquez, Andrés Cárdenas y Deivis Quispe
## Grupo 3
## Fecha: 02-02-2026


# Archivo Laboratorio 6
## Ejercicio 1
Crear una nueva table para almacenar información de nombres y salarios de los empleados

In [38]:
try:
    try:
        cursor.execute("DROP TABLE top_salaries")
    except oracledb.Error:
        pass

    cursor.execute("""
        CREATE TABLE top_salaries (
            emp_name VARCHAR2(100),
            salary   NUMBER(8,2)
        )
    """)
    print("Tabla TOP_SALARIES creada.")

except oracledb.Error as e:
    print(f"Error en DDL: {e}")

Tabla TOP_SALARIES creada.


## Ejercicio 2
Crear un pl/sql que determine el top n del salario de los empleados
a. Solicitar por teclado el valor n que represente el top de los salarios
b. Mediante el uso de cursores en la tabla employees busque el top n de salarios
c. Guarde los nombres y los salarios top en la nueva tabla creada

In [39]:
try:
    n_valor = 5

    cursor.execute(f"""
    DECLARE
        CURSOR c_top_emp IS
            SELECT last_name, salary
            FROM employees
            ORDER BY salary DESC;

        v_name   employees.last_name%TYPE;
        v_salary employees.salary%TYPE;
        v_limit  NUMBER := {n_valor};
    BEGIN
        OPEN c_top_emp;
        LOOP
            FETCH c_top_emp INTO v_name, v_salary;
            EXIT WHEN c_top_emp%NOTFOUND OR c_top_emp%ROWCOUNT > v_limit;

            INSERT INTO top_salaries (emp_name, salary)
            VALUES (v_name, v_salary);

        END LOOP;
        CLOSE c_top_emp;
        COMMIT;
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

## Ejercicio 3
Crear un pl/sql que realice lo siguiente:
a. Utilizando cursores verifique el salario de todos los empleados
b. Si el salario del empleado es menor a $2000 enviar el mensaje siguiente por ejemplo “El empleado SCOTT necesita aumento de sueldo”
c. Si el salario del empleado esta entre $2001 y $3000 enviar el mensaje siguiente por ejemplo “El empleado SCOTT tiene un sueldo aceptable”
d. Si el salario del empleado es mayor a $3001 enviar el mensaje siguiente por ejemplo “El empleado SCOTT no necesita aumento de sueldo”

In [40]:
try:
    cursor.execute(f"""
    DECLARE
        CURSOR c_all_emp IS
            SELECT last_name, salary FROM employees;

        v_emp_rec c_all_emp%ROWTYPE;
    BEGIN
        OPEN c_all_emp;
        LOOP
            FETCH c_all_emp INTO v_emp_rec;
            EXIT WHEN c_all_emp%NOTFOUND;

            IF v_emp_rec.salary < 2000 THEN
                DBMS_OUTPUT.PUT_LINE('El empleado ' || v_emp_rec.last_name || ' necesita aumento de sueldo');
            ELSIF v_emp_rec.salary BETWEEN 2001 AND 3000 THEN
                DBMS_OUTPUT.PUT_LINE('El empleado ' || v_emp_rec.last_name || ' tiene un sueldo aceptable');
            ELSIF v_emp_rec.salary > 3001 THEN
                DBMS_OUTPUT.PUT_LINE('El empleado ' || v_emp_rec.last_name || ' no necesita aumento de sueldo');
            END IF;

        END LOOP;
        CLOSE c_all_emp;
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El empleado King no necesita aumento de sueldo
El empleado Kochhar no necesita aumento de sueldo
El empleado De Haan no necesita aumento de sueldo
El empleado Hunold no necesita aumento de sueldo
El empleado Ernst no necesita aumento de sueldo
El empleado Austin no necesita aumento de sueldo
El empleado Pataballa no necesita aumento de sueldo
El empleado Lorentz no necesita aumento de sueldo
El empleado Greenberg no necesita aumento de sueldo
El empleado Faviet no necesita aumento de sueldo
El empleado Chen no necesita aumento de sueldo
El empleado Sciarra no necesita aumento de sueldo
El empleado Urman no necesita aumento de sueldo
El empleado Popp no necesita aumento de sueldo
El empleado Raphaely no necesita aumento de sueldo
El empleado Khoo no necesita aumento de sueldo
El empleado Baida tiene un sueldo aceptable
El empleado Tobias tiene un sueldo aceptable
El empleado Himuro tiene un sueldo aceptable
El empleado Colmenares tiene un sueldo aceptable
El empleado Weiss no necesita a

# Archivo Laboratorio 7
## Ejercicio 1
Mediante un pl/sql utilice un cursor para recuperar el número, nombre de departamento de la tabla DEPARTMENTS para aquellos códigos de departamento menores a 100. Pasar el numero de departamento a otro cursor para recuperar detalles de los empleados de la tabla EMPLOYEES como el job, hire date, salario de los empleados cuyo código de empleado es menor a 120 y trabajan en ese código de departamento


In [41]:
try:
    cursor.execute(f"""
    DECLARE
        CURSOR c_dept IS
            SELECT department_id, department_name
            FROM departments
            WHERE department_id < 100
            ORDER BY department_id;

        CURSOR c_emp (p_dept_id NUMBER) IS
            SELECT first_name, last_name, job_id, hire_date, salary
            FROM employees
            WHERE employee_id < 120
            AND department_id = p_dept_id;

        v_dept_rec c_dept%ROWTYPE;
        v_emp_rec  c_emp%ROWTYPE;

    BEGIN
        OPEN c_dept;
        LOOP
            FETCH c_dept INTO v_dept_rec;
            EXIT WHEN c_dept%NOTFOUND;

            DBMS_OUTPUT.PUT_LINE('Departamento: ' || v_dept_rec.department_name);

            OPEN c_emp(v_dept_rec.department_id);
            LOOP
                FETCH c_emp INTO v_emp_rec;
                EXIT WHEN c_emp%NOTFOUND;

                DBMS_OUTPUT.PUT_LINE('Empleado: ' || v_emp_rec.first_name || ' ' || v_emp_rec.last_name);
            END LOOP;
            CLOSE c_emp;

        END LOOP;
        CLOSE c_dept;
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Departamento: Administration
Departamento: Marketing
Departamento: Purchasing
Empleado: Den Raphaely
Empleado: Alexander Khoo
Empleado: Shelli Baida
Empleado: Sigal Tobias
Empleado: Guy Himuro
Empleado: Karen Colmenares
Departamento: Human Resources
Departamento: Shipping
Departamento: IT
Empleado: Alexander Hunold
Empleado: Bruce Ernst
Empleado: David Austin
Empleado: Valli Pataballa
Empleado: Diana Lorentz
Departamento: Public Relations
Departamento: Sales
Departamento: Executive
Empleado: Steven King
Empleado: Neena Kochhar
Empleado: Lex De Haan
Departamento: w
Empleado: Nancy Greenberg
Empleado: Daniel Faviet
Empleado: John Chen
Empleado: Ismael Sciarra
Empleado: Jose Manuel Urman
Empleado: Luis Popp
Departamento: Accounting


## Ejercicio 2
Crear un pl/sql incorporando un cursor donde utilice las funcionalidad FOR UPDATE y WHERE CURRENT OF para realizar lo siguiente:
DEFINE p_empno=104
DEFINE p_empno=174
DEFINE p_empno=176

In [42]:
try:
    try:
        cursor.execute("DROP TABLE emp_stars_lab")
    except:
        pass

    cursor.execute("""
        CREATE TABLE emp_stars_lab AS
        SELECT employee_id, salary, cast(NULL as VARCHAR2(50)) as stars
        FROM employees
        WHERE employee_id IN (104, 174, 176)
    """)

    emp_ids_str = "104, 174, 176"

    cursor.execute(f"""
    DECLARE
        CURSOR c_stars IS
            SELECT employee_id, salary, stars
            FROM emp_stars_lab
            WHERE employee_id IN ({emp_ids_str})
            FOR UPDATE OF stars;

        v_emp_id  emp_stars_lab.employee_id%TYPE;
        v_salary  emp_stars_lab.salary%TYPE;
        v_stars   emp_stars_lab.stars%TYPE;

        v_num_stars NUMBER;
        v_bar       VARCHAR2(50);
    BEGIN
        OPEN c_stars;
        LOOP
            FETCH c_stars INTO v_emp_id, v_salary, v_stars;
            EXIT WHEN c_stars%NOTFOUND;

            v_num_stars := TRUNC(v_salary / 1000);
            v_bar := RPAD('*', v_num_stars, '*');

            UPDATE emp_stars_lab
            SET stars = v_bar
            WHERE CURRENT OF c_stars;

            DBMS_OUTPUT.PUT_LINE(v_emp_id || ' ' || v_salary || ' ' || v_bar);

        END LOOP;
        CLOSE c_stars;
        COMMIT;
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

104 6000 ******
176 8600 ********
